# PDF Processing với pdfminer.six

Notebook này sử dụng thư viện **pdfminer.six** để đọc và trích xuất văn bản từ file PDF.

- `pdfminer.six` cung cấp khả năng phân tích cú pháp PDF chi tiết hơn, hỗ trợ tốt tiếng Việt và các ngôn ngữ Unicode.
- Có thể tùy chỉnh cách layout, khoảng cách ký tự, v.v.

In [3]:

!pip install pdfminer.six
!pip install pyvi underthesea
!pip install spacy


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Obtaining dependency information for spacy from https://files.pythonhosted.org/packages/5f/55/4371413a6dfc1fa837282a365498165f828c2f3fe018dfb35336acc869e0/spacy-3.8.14-cp312-cp312-win_amd64.whl.metadata
  Obtaining dependency information for spacy-legacy<3.1.0,>=3.0.11 from https://files.pythonhosted.org/packages/c3/55/12e842c70ff8828e34e543a2c7176dac4da006ca6901c9e8b43efab8bc6b/spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata
  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Obtaining dependency information for spacy-loggers<2.0.0,>=1.0.0 from https://files.pythonhosted.org/packages/33/78/d1a1a026ef3af911159398c939b1509d5c36fe524c7b644f34a5146c4e16/spacy_loggers-1.0.5-py3-none-any.whl.metadata
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Obtaining dependency information for murmurhash<1.1.0,>=0.28.0 from https://files.pythonhosted.org/packages/84/a4/b249b042f5afe34d14ada2dc4afc777e883c15863296756179652e081c44/murmurhash-1.0.15-cp3


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import os
from io import StringIO
import spacy
import re

from pdfminer.high_level import extract_text, extract_pages
from pdfminer.layout import LAParams, LTTextBox, LTTextLine, LTFigure
from pdfminer.pdfpage import PDFPage
from pdfminer.pdfinterp import PDFResourceManager, PDFPageInterpreter
from pdfminer.converter import TextConverter



In [10]:
def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Trích xuất toàn bộ văn bản từ file PDF sử dụng hàm high-level của pdfminer.
    """
    # Khởi tạo tham số cấu trúc layout
    laparams = LAParams(
        line_overlap=0.5,
        char_margin=2.0,
        line_margin=0.5,
        word_margin=0.1,
        boxes_flow=0.5,
        detect_vertical=False,
    )
    
    # Sử dụng hàm extract_text tích hợp sẵn cực kỳ gọn
    text = extract_text(pdf_path, laparams=laparams)
    return text



In [11]:

pdf_path = r"D:/nam_4/explore_bigdata/folder_github/Project-Financial-NLP_Analytics/data/View_Annual_Report.pdf"
txt_path = r"D:/nam_4/explore_bigdata/folder_github/Project-Financial-NLP_Analytics/data/View_Annual_Report.txt"
clean_txt_path = r"D:/nam_4/explore_bigdata/folder_github/Project-Financial-NLP_Analytics/data/View_Annual_Report_Cleaned.txt"

text = extract_text_from_pdf(pdf_path)


with open(txt_path, "r", encoding="utf-8") as f:
    raw_lines = f.read().splitlines()

cleaned_lines = [line.strip() for line in raw_lines if line.strip() != '']


full_text = '\n'.join(cleaned_lines)

# 1. Nối các từ bị ngắt dòng do căn lề (VD: "finan- \n cial" -> "financial")
full_text = re.sub(r'([a-zA-Z]+)-\s*\n\s*([a-zA-Z]+)', r'\1\2', full_text)

# 2. Xóa các dải dấu chấm của mục lục (VD: "Net Sales ........... 100")
full_text = re.sub(r'\.{2,}', ' ', full_text)

# 3. Xóa các con số trơ trọi trên một dòng (thường là số trang bị lọt vào)
full_text = re.sub(r'^\s*\d+\s*$', '', full_text, flags=re.MULTILINE)

# 4. Gom các khoảng trắng dài bất thường thành 1 khoảng trắng duy nhất
full_text = re.sub(r'[^\S\n]{2,}', ' ', full_text)


with open(clean_txt_path, "w", encoding="utf-8") as f:
    f.write(full_text)

print(full_text[:500])




UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
Form 10-K
(Mark One)
ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended September 24, 2016
or
For the transition period from to
Commission File Number: 001-36743
Apple Inc.
(Exact name of Registrant as specified in its charter)
California
(State or other jurisdiction of incorpor


## Tokenization với spaCy cho Tiếng Việt
Khác với tiếng Anh, spaCy không cung cấp sẵn các pre-trained statistical pipeline đầy đủ cho Tiếng Việt. Tuy nhiên, ta hoàn toàn có thể khởi tạo một model ngôn ngữ Tiếng Việt trống (`blank model`) hỗ trợ bởi các tokenizer tiếng Việt (như Pyvi) để tách từ chính xác thay vì tách theo khoảng trắng thông thường.

In [ ]:

nlp = spacy.load("en_core_web_sm")

nlp.max_length = len(full_text) + 100000 

doc = nlp(full_text)


processed_tokens = []

for token in doc:
    if not token.is_punct and not token.is_space and not token.is_stop:
        
        lemma_word = token.lemma_.lower()
        
        if lemma_word.isalpha(): 
            processed_tokens.append(lemma_word)

nlp_txt_path = r"D:/nam_4/explore_bigdata/folder_github/Project-Financial-NLP_Analytics/data/View_Annual_Report_NLP_Tokens.txt"

# Nối danh sách các từ thành một chuỗi văn bản dài
final_nlp_text = ' '.join(processed_tokens)

with open(nlp_txt_path, "w", encoding="utf-8") as f:
    f.write(final_nlp_text)

print(f"✅ Đã xử lý NLP thành công và lưu tại: {nlp_txt_path}")
print(f"📊 Tổng số lượng từ có giá trị thu được: {len(processed_tokens)} từ.")

# In thử 50 từ đầu tiên để kiểm chứng
print("\n--- XEM TRƯỚC 50 TỪ ĐÃ LEMMATIZE ---")
print(processed_tokens[:50])